# 5. Hafta — Makine Öğrenmesine Giriş

**BTP271 Uygulamalı Yapay Zeka** · Ön-Yüz Yazılım Geliştirme

---
## 1. Kural yazmak mı, öğrenmek mi?

Geleneksel programlamada kuralları **biz** yazarız. Aşağıdaki filtre, mesajda `kazandınız`, `bedava` veya `tıklayın` geçiyorsa spam der.

▶️ **Çalıştır** ve ❌ işaretlerine bak.

In [ ]:
def spam_mi(mesaj):
    yasakli = ["kazandınız", "bedava", "tıklayın"]
    for kelime in yasakli:
        if kelime in mesaj.lower():
            return True
    return False

mesajlar = [
    ("Tebrikler, iPhone kazandınız! Hemen tıklayın.", True),
    ("Yarınki ders 13.00'te G blokta.", False),
    ("Bedava kargo fırsatını kaçırmayın!", True),
    ("Toplantı notlarını görmek için linke tıklayın.", False),
    ("T3br1kler! 0dül sizi bekliyor, hemen başvurun.", True),
]

dogru = 0
for mesaj, gercek in mesajlar:
    tahmin = spam_mi(mesaj)
    if tahmin == gercek:
        print("✅", mesaj)
        dogru += 1
    else:
        print("❌", mesaj)

print("Doğruluk:", dogru, "/", len(mesajlar))

**İki hata var:**
- **Yanlış alarm:** Normal bir iş mesajı "tıklayın" içerdiği için spam sanıldı.
- **Kaçan spam:** Harfleri değiştirilmiş mesaj hiçbir kurala takılmadı.

Her yeni istisna için yeni kural yazmak gerekir ve bu liste hiç bitmez.

**Makine öğrenmesinde** kuralları biz yazmayız: bilgisayara **etiketli örnekler** veririz, kuralı **kendisi bulur**.

| Geleneksel programlama | Makine öğrenmesi |
|---|---|
| Kurallar + Veri → Program → Cevaplar | Veri + Doğru cevaplar → Öğrenme algoritması → Kurallar (model) |

#### ✍️ Kendin Dene
Aşağıdaki `...` yerine kendi yazdığın bir mesajı yaz ve çalıştır. Kural tabanlı filtre onu yakalıyor mu?

In [ ]:
spam_mi("______")

---
## 2. Temel kavramlar: örnek, öznitelik, etiket


| Kavram | Tabloda nerede? | Iris örneği |
|---|---|---|
| **Örnek** | Bir satır | Bir çiçek |
| **Öznitelik (X)** | Modelin baktığı sütunlar | Yaprak ölçüleri |
| **Etiket (y)** | Tahmin edilecek sütun | Tür |

Tahmin edilen şey **sayı** ise → **regresyon**. **Kategori** ise → **sınıflandırma**.

▶️ **Çalıştır:** Iris verisini yükle ve X ile y'yi ayır.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris

iris = load_iris(as_frame=True)
df_iris = iris.frame
df_iris["tur"] = df_iris["target"].map({0: "setosa", 1: "versicolor", 2: "virginica"})

X = df_iris[["petal length (cm)", "petal width (cm)"]]   # öznitelikler
y = df_iris["tur"]                                       # etiket

print("X:", X.shape, "→ 150 çiçek, 2 öznitelik")
print("y:", y.shape, "→ 150 etiket")
df_iris.head()

#### ✍️ Kendin Yaz — Ev verisinde X ve y
Önce aşağıdaki hücreyi **çalıştır**

In [ ]:
csv_metni = """ilan_id,semt,metrekare,oda_sayisi,bina_yasi,kat,fiyat
1,Avcılar,95,2,12,3,4250000
2,Avcılar,120,3,5,2,5900000
3,Beylikdüzü,140,3,3,7,6800000
4,Kadıköy,85,2,25,4,9500000
5,Beşiktaş,110,3,30,2,13200000
6,Esenyurt,100,2,2,5,3600000
7,Avcılar,150,4,,1,7100000
8,Kadıköy,130,3,15,3,14500000
9,Beylikdüzü,90,2,8,10,4100000
10,Esenyurt,135,3,4,8,4700000
11,Beşiktaş,75,1,40,1,
12,Avcılar,110,3,20,4,5200000
13,Kadıköy,160,4,,6,19800000
14,Beylikdüzü,180,4,1,3,8900000
15,Esenyurt,80,1,6,2,
9,Beylikdüzü,90,2,8,10,4100000
"""
with open("evler.csv", "w", encoding="utf-8") as f:
    f.write(csv_metni)

df = pd.read_csv("evler.csv")
temiz = df.drop_duplicates().dropna(subset=["fiyat"])
temiz = temiz.fillna({"bina_yasi": temiz["bina_yasi"].median()})
temiz.head()

**Görev:** Model; evin **metrekare, oda sayısı ve bina yaşına** bakıp **fiyatını** tahmin edecek.

- `X_ev` → modelin baktığı 3 sütun
- `y_ev` → tahmin edilecek sütun

> 🧩 **İpucu:** `y` tek sütundur, tahmin edilecek olan şeydir. Sütunun adı tablonun en sağında.

In [ ]:
X_ev = temiz[["metrekare", "oda_sayisi", "bina_yasi"]]
y_ev = temiz["____"]

print(X_ev.shape)
print(y_ev.shape)

# Bu problem regresyon mu, sınıflandırma mı? Aşağıya yaz:
cevap = "_______"

In [ ]:
# 🧪 Test
assert list(X_ev.columns) == ["metrekare", "oda_sayisi", "bina_yasi"], "X_ev'de 3 sütun olmalı"
assert y_ev.name == "fiyat", "y_ev fiyat sütunu olmalı"
assert cevap.lower().startswith("regresyon"), "Fiyat bir SAYI. Sayı tahmini hangi türdü?"
print("✅ Tüm testler geçti!")

### Kategorik sütunlar: modeller metin anlamaz


`semt` sütunu metindir ("Avcılar", "Kadıköy"). Modeller sadece **sayıyla** çalışır, bu yüzden metni sayıya çeviririz.

- **One-hot encoding:** Her semt için ayrı bir 0/1 sütunu açar. (Sırası olmayan girdiler için)
- **Label encoding:** Her kategoriye bir sayı verir: 0, 1, 2… (Hedef etiket y için)

▶️ **Çalıştır** ve her satırda sadece bir tane `1` olduğuna bak.

In [ ]:
kodlu = pd.get_dummies(temiz["semt"], prefix="semt", dtype=int)
kodlu.head()

> ⚠️ Semte label encoding yapsaydık (Avcılar=0 … Kadıköy=4), model "Kadıköy, Avcılar'dan 4 kat büyük" gibi **sahte bir sıra** öğrenebilirdi. Semt gibi sırasız girdilerde bu yüzden one-hot kullanırız.

▶️ **Çalıştır:** Iris'te tür adları zaten 0, 1, 2 olarak kodlanmış (label encoding).

In [ ]:
df_iris[["tur", "target"]].drop_duplicates()

---
## 3. Denetimli öğrenme: kuralı veriden bulmak


Denetimli öğrenmede modele hem **X** hem de **doğru cevapları (y)** veririz. Model kuralı kendisi öğrenir.

▶️ **Çalıştır:** Taç yaprağı uzunluğunun türlere göre dağılımına bak. Setosa nerede duruyor?

In [ ]:
for tur, grup in df_iris.groupby("tur"):
    plt.hist(grup["petal length (cm)"], bins=10, alpha=0.7, label=tur)

plt.title("Taç yaprağı uzunluğu (türlere göre)")
plt.xlabel("Taç yaprağı uzunluğu (cm)")
plt.ylabel("Çiçek sayısı")
plt.legend()
plt.show()

### Elle yazılmış kural ve doğruluk
Setosa, 2.5 cm'nin altında kalıyor. Kuralımız: **"Uzunluk < 2.5 ise setosa."**

**Doğruluk(accuracy) = doğru tahmin sayısı ÷ toplam tahmin sayısı.**

▶️ **Çalıştır:** İlk 5 çiçeğe bakalım. `tahmin` bizim tahminimiz, `gercek` doğru cevap.

In [ ]:
tahmin = df_iris["petal length (cm)"] < 2.5     # "setosa mı?" tahmini
gercek = df_iris["tur"] == "_______"              # gerçek cevap

# Tahmin ile gerçek aynı mı? (True = doğru bildik, False = yanlış bildik)
dogru_mu = (tahmin == gercek)

print("İlk 5 çiçek için doğru mu?")
print(dogru_mu.head())

Python'da **True = 1, False = 0** sayılır. O zaman doğru sayısını toplayıp çiçek sayısına bölersek doğruluğu buluruz.

▶️ **Çalıştır**

In [ ]:
dogru_sayisi = dogru_mu.sum()          # True'ları say
toplam = len(dogru_mu)                 # toplam çiçek sayısı

print("Doğru bilinen:", dogru_sayisi, "/", toplam)
print(f"Doğruluk: %{dogru_sayisi / toplam * 100:.1f}")

# .mean() aynı hesabı tek adımda yapar
dogruluk = dogru_mu.mean()
print(f"Doğruluk (.mean ile): %{dogruluk * 100:.1f}")

### Eşiği bilgisayar bulsun
2.5'i **biz** seçtik. Şimdi bilgisayar **birkaç eşiği** denesin.

> 🔁 **`for` döngüsü:** `for esik in [1.0, 2.0, 2.5]:` demek, "`esik` önce 1.0 olsun, alttaki satırları çalıştır; sonra 2.0 olsun, tekrar çalıştır; sonra 2.5..." demektir.

▶️ **Çalıştır:** Hangi eşik en iyi?

In [ ]:
for esik in [1.0, 2.0, 2.5, 3.0, 5.0]:
    tahmin = df_iris["petal length (cm)"] < esik
    dogruluk = (tahmin == gercek).mean()
    print(f"Eşik {esik} → doğruluk %{dogruluk * 100:.1f}")

Şimdi bilgisayar 1.0'dan 7.0'a kadar **60 eşiğin hepsini** denesin ve en iyisini **hatırlasın**.
**Öğrenmenin en basit hali budur: en iyi kuralı aramak.**



In [ ]:
en_iyi_esik = None      # şimdilik bilmiyoruz
en_iyi_dogruluk = 0     # şimdilik en iyi doğruluk 0

for esik in np.arange(1.0, 7.0, 0.1):                    # 1.0, 1.1, 1.2 ... 6.9
    esik = round(esik, 1)                                # 2.0000001 gibi sayıları düzelt
    tahmin = df_iris["petal length (cm)"] < esik         # bu eşikle tahmin yap
    dogruluk = (tahmin == gercek).mean()                 # doğruluğu hesapla
    if dogruluk > en_iyi_dogruluk:                       # önceki en iyiden iyiyse
        en_iyi_esik = esik                               # eşiği hatırla
        en_iyi_dogruluk = dogruluk                       # doğruluğu hatırla

print(f"Bulunan eşik: {en_iyi_esik} cm | Doğruluk: %{en_iyi_dogruluk * 100:.1f}")

---
### ✍️ Kendin Yaz —  versicolor ve virginica
Şimdi birbirine yakın iki türü ayıran eşiği bulalım.
Kural: **"Uzunluk > eşik ise virginica."**

Dört adımda yapacağız.

**Adım 1:** Setosa'ları çıkar. Kalan 100 çiçek `iki_tur` olsun.

> 🧩 **İpucu:** `!=` "eşit değilse" demektir. Hangi türü çıkarıyoruz?

In [ ]:
iki_tur = df_iris[df_iris["tur"] != "________"]
gercek2 = iki_tur["tur"] == "virginica"      # gerçek cevap: virginica mı?

print(len(iki_tur), "çiçek kaldı")

**Adım 2:** Tek bir eşiği (4.7) elle dene.

> 🧩 **İpucu:** Doğruluğu hesaplarken tahmini **gerçek cevapla** karşılaştırırız. Gerçek cevap değişkeninin adı bir önceki hücrede.

In [ ]:
tahmin2 = iki_tur["petal length (cm)"] > 4.7
dogruluk2 = (tahmin2 == ___).mean()

print(f"Eşik 4.7 → doğruluk %{dogruluk2 * 100:.1f}")

**Adım 3:** Şimdi 3.0'dan 7.0'a kadar **tüm eşikleri** dene. Her eşiği ve doğruluğunu iki listeye ekle.

> 🧩 **İpucu:** Döngüdeki değişkenin adı `esik`. Eşik olarak onu kullan.

In [ ]:
esikler = []         # denenen eşikler
dogruluklar = []     # her eşiğin doğruluğu

for esik in np.arange(3.0, 7.0, 0.1):
    esik = round(esik, 1)
    tahmin2 = iki_tur["petal length (cm)"] > ___
    dogruluk = (tahmin2 == gercek2).mean()
    esikler.append(esik)
    dogruluklar.append(dogruluk)

print(len(esikler), "eşik denendi")

**Adım 4:** Doğrulukların içinden **en yükseğini** bul. Onun eşiğini bulan ikinci satır hazır.

> 🧩 **İpucu:** `max(liste)` listedeki **en büyük** sayıyı verir. Örnek: `max([3, 9, 5])` → `9`.
> İkinci satır, bu en yüksek doğruluğun listede kaçıncı sırada olduğuna bakıp aynı sıradaki eşiği alır. Onu anlamak zorunda değilsin.

In [ ]:
en_iyi_dogruluk2 = ___(dogruluklar)
en_iyi_esik2 = esikler[dogruluklar.index(en_iyi_dogruluk2)]

print(f"En iyi eşik: {en_iyi_esik2} cm | Doğruluk: %{en_iyi_dogruluk2 * 100:.1f}")

▶️ **Çalıştır:** Grafiğe bak. Doğruluk neden hiçbir zaman %100'e çıkmıyor?

In [ ]:
plt.plot(esikler, dogruluklar)
plt.axvline(en_iyi_esik2, color="red", linestyle="--", label=f"En iyi eşik: {en_iyi_esik2}")
plt.title("Eşik değerine göre doğruluk")
plt.xlabel("Eşik (cm)")
plt.ylabel("Doğruluk")
plt.legend()
plt.show()

**Soru:** Doğruluk neden %100'e ulaşmıyor? Aşağıdaki cümleyi tamamla.

> 🧩 **İpucu:** Sunumdaki grafikte iki türün noktaları aynı bölgede ne yapıyordu?

In [ ]:
cevap2 = "İki türün uzunlukları bir bölgede ___, tek çizgi hepsini ayıramaz."

In [ ]:
# 🧪 Test
assert len(iki_tur) == 100, "Adım 1: Setosa çıkarılınca 100 çiçek kalmalı"
assert len(esikler) == len(dogruluklar) == 40, "Adım 3: 40 eşik denenmeli"
assert en_iyi_esik2 == 4.7, "Adım 4: En iyi eşik 4.7 çıkmalı"
assert abs(en_iyi_dogruluk2 - 0.93) < 1e-9, "Adım 4: Doğruluk %93 çıkmalı"
print("✅ Tüm testler geçti!")

---
## 4. Denetimsiz öğrenme: etiket yokken


Burada **doğru cevap (y) yok**. Model sadece X'e bakar ve birbirine benzeyen çiçekleri **gruplar** (kümeleme).

**K-Means**'e sadece ölçüleri veriyoruz, türleri **göstermiyoruz**.

▶️ **Çalıştır ve izle**

In [ ]:
from sklearn.cluster import KMeans

model = KMeans(n_clusters=3, n_init=10, random_state=42)   # "3 grup bul"
df_iris["kume"] = model.fit_predict(X)                      # sadece X, y yok!

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for tur, grup in df_iris.groupby("tur"):
    axes[0].scatter(grup["petal length (cm)"], grup["petal width (cm)"], label=tur, s=30)
axes[0].set_title("Gerçek türler (model bunları GÖRMEDİ)")
axes[0].legend()

for kume, grup in df_iris.groupby("kume"):
    axes[1].scatter(grup["petal length (cm)"], grup["petal width (cm)"], label=f"Küme {kume}", s=30)
axes[1].set_title("K-Means'in bulduğu kümeler")
axes[1].legend()

for ax in axes:
    ax.set_xlabel("Taç yaprağı uzunluğu (cm)")
    ax.set_ylabel("Taç yaprağı genişliği (cm)")
plt.tight_layout()
plt.show()

> Renkler eşleşmiyor ama bu **normal**: küme numaralarının (0, 1, 2) anlamı yok, model türlerin adını bilmiyor.

#### ✍️ Kendin Yaz — Kümeler türlerle örtüşüyor mu?
`pd.crosstab(a, b)` iki sütunu çapraz sayar: hangi türden kaç çiçek hangi kümeye düşmüş?

> 🧩 **İpucu:** Bir önceki hücrede küme numaralarını `df_iris["kume"]` diye kaydetmiştik. Boşluğa sütunun adını yaz: `kume`.

In [ ]:
capraz = pd.crosstab(df_iris["tur"], df_iris["kume"])
capraz

Tabloya bak ve cevapla: **50 setosa'nın hepsi tek bir kümede mi?** (evet / hayır)

In [ ]:
cevap3 = "___"

In [ ]:
# 🧪 Test
assert capraz.shape == (3, 3), "Tablo 3 satır x 3 sütun olmalı"
assert capraz.values.sum() == 150, "Toplam 150 çiçek olmalı"
assert cevap3.lower() == "evet", "Setosa satırına bak: 50 sayısı tek bir sütunda mı?"
print("✅ Tüm testler geçti!")

---
## 5. Pekiştirmeli öğrenme: deneme, yanılma, ödül

Hazır veri yok. **Ajan** denedikçe öğrenir.

**Örnek:** Sitede 3 farklı "Satın Al" butonu var (A, B, C). Ajan her ziyaretçiye bir buton gösterir. Tıklanırsa **ödül = 1**, tıklanmazsa 0.

İkilem:
- **Keşfet:** Az denenen butonları da göster, belki daha iyidir.
- **Sömür:** Şimdiye kadar en iyi görüneni göster.

`epsilon` = keşfetme oranı. `0.1` → ziyaretçilerin %10'unda rastgele buton.

Gerçek tıklanma oranları: **A %20, B %50, C %35** (ajan bunları bilmiyor).

▶️ **Çalıştır:** Aşağıdaki `simule_et` fonksiyonu bir **kara kutu**. İçini anlaman gerekmiyor. Sadece `epsilon` verirsin, 1000 ziyaretçilik simülasyonu çalıştırıp sonucu verir.

In [ ]:
import random

def simule_et(epsilon, tur=1000, tohum=42):
    """Döndürür: toplam tıklama, ajanın öğrendiği oranlar, her butonun gösterim sayısı"""
    random.seed(tohum)
    oranlar = {"A": 0.20, "B": 0.50, "C": 0.35}      # gerçek oranlar (ajan BİLMİYOR)
    tahmin = {b: 0.0 for b in oranlar}
    secim = {b: 0 for b in oranlar}
    toplam_odul = 0

    for _ in range(tur):
        if random.random() < epsilon:
            buton = random.choice(list(oranlar))     # keşfet: rastgele buton
        else:
            buton = max(tahmin, key=tahmin.get)      # sömür: en iyi bilinen buton
        odul = 1 if random.random() < oranlar[buton] else 0
        secim[buton] += 1
        tahmin[buton] += (odul - tahmin[buton]) / secim[buton]
        toplam_odul += odul

    return toplam_odul, tahmin, secim

odul, tahmin, secim = simule_et(0.1)
print("Toplam tıklama:", odul, "/ 1000 ziyaretçi")
print("Hangi buton kaç kez gösterildi:", secim)

#### ✍️ Kendin Yaz — Keşfetmek ne kadar önemli?
`epsilon` için üç değer deneyeceğiz: **0** (hiç keşfetme), **0.1** (biraz keşfet), **1** (hep rastgele).

> 🧩 **İpucu:** `simule_et` fonksiyonuna döngüdeki `eps` değerini ver. Diğer satırlar hazır.

In [ ]:
sonuclar = {}

for eps in [0, 0.1, 1]:
    odul, tahmin, secim = simule_et(___)
    sonuclar[eps] = odul
    print(f"epsilon={eps}: {odul} tıklama | gösterimler: {secim}")

▶️ **Çalıştır:** Grafiğe bak.

In [ ]:
plt.bar([str(e) for e in sonuclar], list(sonuclar.values()))
plt.title("Keşif oranına göre toplam tıklama (1000 ziyaretçi)")
plt.xlabel("epsilon")
plt.ylabel("Toplam tıklama")
plt.show()

**Soru:** Hangi epsilon en çok tıklamayı getirdi? (`0`, `0.1` veya `1`)

> 🧩 **İpucu:** Sunumdaki "Keşfetmek ne kadar önemli?" tablosuna bak. Ne çok keşfet, ne hiç keşfetme.

In [ ]:
en_iyi_epsilon = ___

In [ ]:
# 🧪 Test
assert set(sonuclar) == {0, 0.1, 1}, "sonuclar'da 0, 0.1 ve 1 olmalı"
assert sonuclar[0.1] > sonuclar[1] > sonuclar[0], "Sonuçlar beklenenden farklı, simule_et(eps) ve odul'u kontrol et"
assert en_iyi_epsilon == 0.1, "Grafikte en yüksek çubuk hangisi?"
print("✅ Tüm testler geçti!")

**Özet:**
- `epsilon = 0` → Ajan hiç keşfetmedi, ilk denediği A butonunda takıldı. B'nin daha iyi olduğunu hiç öğrenemedi.
- `epsilon = 1` → Ajan hep rastgele davrandı, B'nin iyi olduğunu öğrense bile kullanmadı.
- `epsilon = 0.1` → **Denge:** biraz keşfet, çoğunlukla bildiğini uygula.

---
## 🏁 Haftanın Mini Projesi: Kendi Iris Sınıflandırıcın


Hazır model kullanmadan, **veriden öğrenen** ve **üç türü birden** tahmin eden bir sınıflandırıcı yazacaksın. Hedef: **%96 doğruluk**.

Fikir basit: **iki kural** art arda kullanılır.
1. Uzunluk 2.5'ten küçükse → **setosa**
2. Değilse, genişlik bir eşikten büyükse → **virginica**, değilse → **versicolor**

### Adım 1: Genişlik eşiğini bul
Daha önce *uzunluk* için yaptığımızın aynısını bu kez **genişlik** için yap. Kural: **genişlik > eşik ise virginica.**

> 🧩 **İpucu:** Adım 3'teki döngünün aynısı. `esik` değişkenini kullan.

In [ ]:
genislik_esigi = None
en_iyi = 0

for esik in np.arange(1.0, 2.6, 0.1):
    esik = round(esik, 1)
    tahmin_g = iki_tur["petal width (cm)"] > ___
    dogruluk = (tahmin_g == gercek2).mean()
    if dogruluk > en_iyi:
        genislik_esigi = esik
        en_iyi = dogruluk

print(f"Genişlik eşiği: {genislik_esigi} (iki türde doğruluk %{en_iyi * 100:.0f})")

### Adım 2: `tahmin_et` fonksiyonunu yaz
Fonksiyon uzunluk ve genişliği alır, tür adını döndürür.

> 🧩 **İpucu:**
> - Setosa kuralındaki sayı 2.5.
> - İkinci kuralda Adım 1'de bulduğun değişkeni kullan: `genislik_esigi`

In [ ]:
def tahmin_et(uzunluk, genislik):
    if uzunluk < ___:
        return "setosa"
    if genislik > ___:
        return "virginica"
    return "versicolor"

# Hızlı deneme:
print(tahmin_et(1.4, 0.2))
print(tahmin_et(5.8, 2.2))

### Adım 3: 150 çiçeğin hepsini test et
Aşağıdaki döngü her çiçek için `tahmin_et` çağırır ve sonuçları yeni bir sütuna yazar. (`zip` iki sütunu yan yana gezer)

> 🧩 **İpucu:** Doğruluk = tahminlerin gerçek türle eşleşme oranı. Gerçek tür sütunu: `"tur"`.

In [ ]:
tahminler = []
for u, g in zip(df_iris["petal length (cm)"], df_iris["petal width (cm)"]):
    tahminler.append(tahmin_et(u, g))

df_iris["tahmin"] = tahminler

genel_dogruluk = (df_iris["tahmin"] == df_iris["___"]).mean()
print(f"Genel doğruluk: %{genel_dogruluk * 100:.1f}")

### Adım 4: Web'den gelen çiçekler → JSON
Bir web formundan 3 yeni çiçek geldi. Hepsini tahmin edip **JSON** olarak yazdıracağız (JavaScript tarafı bunu okuyabilir).

▶️ **Çalıştır**

In [ ]:
import json

yeni_cicekler = [
    {"uzunluk": 1.5, "genislik": 0.3},
    {"uzunluk": 4.2, "genislik": 1.3},
    {"uzunluk": 5.8, "genislik": 2.2},
]

sonuc = []
for c in yeni_cicekler:
    tahmin_tur = tahmin_et(c["uzunluk"], c["genislik"])
    sonuc.append({
        "uzunluk": c["uzunluk"],
        "genislik": c["genislik"],
        "tahmin": tahmin_tur,
    })

print(json.dumps(sonuc, ensure_ascii=False, indent=2))

In [ ]:
# 🧪 Final testi
assert genislik_esigi == 1.6, "Adım 1: Genişlik eşiği 1.6 çıkmalı"
assert tahmin_et(1.4, 0.2) == "setosa", "Adım 2: setosa kuralını kontrol et"
assert tahmin_et(4.2, 1.3) == "versicolor", "Adım 2: versicolor durumunu kontrol et"
assert tahmin_et(5.8, 2.2) == "virginica", "Adım 2: virginica kuralını kontrol et"
assert genel_dogruluk >= 0.95, "Adım 3: Doğruluk en az %95 olmalı"
print("✅ Tüm testler geçti! Tebrikler 🎉 İlk sınıflandırıcını veriden öğrettin.")